In [39]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))

from common import load_matches, load_team_matches, PROCESSED, FIGURES, RESULTS

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

In [40]:
# Load the four feature files produced by the team, and check each one before merging.
prev = pd.read_csv(PROCESSED / "features_previous_performance.csv")   # Arpan
fifa = pd.read_csv(PROCESSED / "features_fifa_rating.csv")            # Nishan (one row per TEAM)
form = pd.read_csv(PROCESSED / "features_recent_form.csv")            # Supriya
ctx  = pd.read_csv(PROCESSED / "features_context_opponent.csv")       # Abhi

for name, df, key in [("previous_performance", prev, ["Match_ID", "Team"]),
                      ("recent_form", form, ["Match_ID", "Team"]),
                      ("context_opponent", ctx, ["Match_ID", "Team"]),
                      ("fifa_rating", fifa, ["Team"])]:
    dup = df.duplicated(key).sum()
    print(f"{name:22s} rows = {len(df):3d} | duplicate keys = {dup}")
    assert dup == 0, f"{name} has duplicate keys"

assert len(prev) == len(form) == len(ctx) == 208 and len(fifa) == 48

previous_performance   rows = 208 | duplicate keys = 0
recent_form            rows = 208 | duplicate keys = 0
context_opponent       rows = 208 | duplicate keys = 0
fifa_rating            rows =  48 | duplicate keys = 0


In [41]:
# Start from the 208 team-match rows (this also supplies the target, Goals_For) and
# attach each part's features. validate="one_to_one" stops silent row duplication.
master = load_team_matches()[["Match_ID", "Round", "Stage_Level", "Team", "Opponent", "Goals_For", "Goals_Against"]]

master = (master
          .merge(prev, on=["Match_ID", "Team"], how="left", validate="one_to_one")
          .merge(form, on=["Match_ID", "Team"], how="left", validate="one_to_one")
          .merge(ctx,  on=["Match_ID", "Team"], how="left", validate="one_to_one")
          .merge(fifa[["Team", "FIFA_Rank", "FIFA_Points"]], on="Team", how="left", validate="many_to_one"))

# The opponent's FIFA points are needed to build the strength gap
opp_fifa = fifa[["Team", "FIFA_Points"]].rename(columns={"Team": "Opponent", "FIFA_Points": "Opp_FIFA_Points"})
master = master.merge(opp_fifa, on="Opponent", how="left", validate="many_to_one")
master["FIFA_Points_Diff"] = master["FIFA_Points"] - master["Opp_FIFA_Points"]

# Rows that are allowed to be empty: the "Raw" averages are undefined for opening matches
allowed_empty = ["Avg_GF_Raw", "Avg_GA_Raw", "Avg_Pts_Raw"]
missing = master.drop(columns=allowed_empty).isna().sum()
print(f"Master table: {master.shape[0]} rows x {master.shape[1]} columns")
print("Unexpected missing values:", int(missing.sum()))
assert len(master) == 208 and missing.sum() == 0

Master table: 208 rows x 30 columns
Unexpected missing values: 0


In [42]:
# ===================================================================
# THE VARIABLE SPLIT  (draft - the team confirms this, then edit this cell only)
# The brief: exactly 8 explanatory variables per task, and at most 4 shared
# between Task 2.1 and Task 2.2. A variable counts as "shared" if it is the
# same underlying quantity, even if 2.1 uses a difference (Team A - Team B)
# and 2.2 uses the team's own value, so the rule is applied strictly.
# Each entry:  concept name -> (column used in Task 2.1, column used in Task 2.2)
# ===================================================================
SHARED = {
    "FIFA strength gap":    ("FIFA_Points_Diff",            "FIFA_Points_Diff"),
    "Stage of tournament":  ("Stage_Level",                 "Stage_Level"),
    "Rest days":            ("Rest_Days_Diff",              "Rest_Days"),
    "Host-country edge":    ("Home_Country_Advantage_Diff", "Home_Country_Advantage"),
}
ONLY_21 = {
    "Same confederation":        "Same_Confederation",
    "Squad age gap":             "Squad_Age_Diff",
    "Scoring form gap":          "Avg_GF_Before_Diff",
    "Defensive form gap":        "Avg_GA_Before_Diff",
}
ONLY_22 = {
    "Matches already played":    "Matches_Played_Before",
    "Last match points":         "Last_Match_Points",
    "Last match goal difference":"Last_Match_Goal_Diff",
    "Opponent squad age":        "Opp_Squad_Age",
}

n21 = len(SHARED) + len(ONLY_21)
n22 = len(SHARED) + len(ONLY_22)
print(f"Task 2.1 variables: {n21} | Task 2.2 variables: {n22} | shared concepts: {len(SHARED)}")
assert n21 == 8 and n22 == 8, "each task needs exactly 8 explanatory variables"
assert len(SHARED) <= 4, "at most 4 variables may be shared between the two tasks"
assert not set(ONLY_21) & set(ONLY_22), "a concept listed as unique to one task also appears in the other"
print("Variable-split rules satisfied.")

Task 2.1 variables: 8 | Task 2.2 variables: 8 | shared concepts: 4
Variable-split rules satisfied.


In [43]:
# ----- TASK 2.2: 208 rows, one team in one match -----
t22 = pd.DataFrame({"Match_ID": master["Match_ID"], "Team": master["Team"], "Opponent": master["Opponent"]})
for concept, (col21, col22) in SHARED.items():
    t22[col22] = master[col22]
for concept, col in ONLY_22.items():
    t22[col] = master[col]
t22["Goals_Scored"] = master["Goals_For"]            # the target

assert len(t22) == 208 and t22.isna().sum().sum() == 0
print(t22.shape)
t22.head(4)

(208, 12)


,Match_ID,Team,Opponent,FIFA_Points_Diff,Stage_Level,Rest_Days,Home_Country_Advantage,Matches_Played_Before,Last_Match_Points,Last_Match_Goal_Diff,Opp_Squad_Age,Goals_Scored
0,19,Algeria,Argentina,-306.24,1,7.0,0,0,1.0,0.0,29.6,0
1,44,Algeria,Jordan,183.29,1,6.0,0,1,0.0,-3.0,28.2,2
2,72,Algeria,Austria,-26.37,1,5.0,0,2,3.0,1.0,28.8,3
3,85,Algeria,Switzerland,-79.03,2,5.0,0,3,1.0,0.0,29.4,0


In [44]:
# ----- TASK 2.1: 104 rows, one match, Team A minus Team B -----
m = load_matches()[["Match_ID", "Team_A", "Team_B", "Goal_Diff", "Stage_Level"]]

side = master.set_index(["Match_ID", "Team"])
A = side.loc[list(zip(m["Match_ID"], m["Team_A"]))].reset_index(drop=True)
B = side.loc[list(zip(m["Match_ID"], m["Team_B"]))].reset_index(drop=True)

t21 = pd.DataFrame({"Match_ID": m["Match_ID"], "Team_A": m["Team_A"], "Team_B": m["Team_B"]})
t21["FIFA_Points_Diff"]             = A["FIFA_Points"] - B["FIFA_Points"]
t21["Stage_Level"]                  = m["Stage_Level"]
t21["Rest_Days_Diff"]               = A["Rest_Days"] - B["Rest_Days"]
t21["Home_Country_Advantage_Diff"]  = A["Home_Country_Advantage"] - B["Home_Country_Advantage"]
t21["Same_Confederation"]           = A["Same_Confederation"]          # identical for A and B
t21["Squad_Age_Diff"]               = A["Squad_Age"] - B["Squad_Age"]
t21["Avg_GF_Before_Diff"]           = A["Avg_GF_Before"] - B["Avg_GF_Before"]
t21["Avg_GA_Before_Diff"]           = A["Avg_GA_Before"] - B["Avg_GA_Before"]
t21["Goal_Diff"]                    = m["Goal_Diff"]                   # the target

# the columns must be exactly the ones the split above promises
expected21 = [c21 for c21, _ in SHARED.values()] + list(ONLY_21.values())
assert [c for c in t21.columns if c not in ("Match_ID", "Team_A", "Team_B", "Goal_Diff")] == expected21
assert len(t21) == 104 and t21.isna().sum().sum() == 0
print(t21.shape)
t21.head(4)

(104, 12)


,Match_ID,Team_A,Team_B,FIFA_Points_Diff,Stage_Level,Rest_Days_Diff,Home_Country_Advantage_Diff,Same_Confederation,Squad_Age_Diff,Avg_GF_Before_Diff,Avg_GA_Before_Diff,Goal_Diff
0,1,Mexico,South Africa,259.10,1,0.0,1,0,0.9,0.0,0.0,2
1,2,Korea Republic,Czechia,85.89,1,0.0,0,0,0.4,0.0,0.0,1
2,3,Canada,Bosnia-Herz,172.26,1,0.0,1,0,0.3,0.0,0.0,0
3,4,USA,Paraguay,165.88,1,0.0,1,0,-1.3,0.0,0.0,3


In [45]:
# ===== Consistency checks between the two datasets (catches merge mistakes) =====
# 1) Total goals: the whole tournament had 308 goals in play, so the Task 2.2 target must sum to 308
total_goals = int(t22["Goals_Scored"].sum())
print("Total goals in Task 2.2 target:", total_goals, "(expected 308)")
assert total_goals == 308

# 2) For every match, Team A's goals minus Team B's goals (from 2.2) must equal Task 2.1's target
g = t22.set_index(["Match_ID", "Team"])["Goals_Scored"]
check = pd.Series([g[(mid, a)] - g[(mid, b)] for mid, a, b in zip(t21.Match_ID, t21.Team_A, t21.Team_B)])
assert (check.values == t21["Goal_Diff"].values).all(), "Task 2.1 target disagrees with Task 2.2 targets"
print("Goal difference in Task 2.1 equals Goals_Scored(A) - Goals_Scored(B) in Task 2.2: OK")

# 3) Variables shared by concept must be genuinely related across the two tables
print("\nShared-variable sanity check (Stage_Level should match exactly):")
stage_chk = t22.merge(t21[["Match_ID", "Stage_Level"]], on="Match_ID", suffixes=("_22", "_21"))
assert (stage_chk["Stage_Level_22"] == stage_chk["Stage_Level_21"]).all()
print("Stage_Level identical in both: OK")

# 4) No look-ahead: opening matches must hold the neutral prior values
opening = master["Matches_Played_Before"] == 0
print(f"\nOpening-match rows: {int(opening.sum())} | their Avg_GF_Before value(s): {master.loc[opening, 'Avg_GF_Before'].unique().tolist()}")
assert opening.sum() == 48

Total goals in Task 2.2 target: 308 (expected 308)
Goal difference in Task 2.1 equals Goals_Scored(A) - Goals_Scored(B) in Task 2.2: OK

Shared-variable sanity check (Stage_Level should match exactly):
Stage_Level identical in both: OK

Opening-match rows: 48 | their Avg_GF_Before value(s): [1.35]


In [46]:
# How many of the 8 variables does each task share? (needed for the report)
shared_cols = [c22 for _, c22 in SHARED.values()]
print("Shared concepts (", len(SHARED), "):", list(SHARED))
print("Only in Task 2.1:", list(ONLY_21))
print("Only in Task 2.2:", list(ONLY_22))

# Quick first look at how each variable relates to its target (Pearson r) - NOT a model, just orientation
r21 = t21.drop(columns=["Match_ID", "Team_A", "Team_B"]).corr()["Goal_Diff"].drop("Goal_Diff").round(2)
r22 = t22.drop(columns=["Match_ID", "Team", "Opponent"]).corr()["Goals_Scored"].drop("Goals_Scored").round(2)
print("\nTask 2.1 - correlation with Goal_Diff:");     print(r21.to_string())
print("\nTask 2.2 - correlation with Goals_Scored:");   print(r22.to_string())

Shared concepts ( 4 ): ['FIFA strength gap', 'Stage of tournament', 'Rest days', 'Host-country edge']
Only in Task 2.1: ['Same confederation', 'Squad age gap', 'Scoring form gap', 'Defensive form gap']
Only in Task 2.2: ['Matches already played', 'Last match points', 'Last match goal difference', 'Opponent squad age']

Task 2.1 - correlation with Goal_Diff:
FIFA_Points_Diff               0.65
Stage_Level                   -0.15
Rest_Days_Diff                -0.05
Home_Country_Advantage_Diff    0.16
Same_Confederation             0.03
Squad_Age_Diff                -0.03
Avg_GF_Before_Diff             0.16
Avg_GA_Before_Diff            -0.33

Task 2.2 - correlation with Goals_Scored:
FIFA_Points_Diff          0.50
Stage_Level               0.03
Rest_Days                -0.05
Home_Country_Advantage    0.14
Matches_Played_Before     0.03
Last_Match_Points         0.02
Last_Match_Goal_Diff      0.08
Opp_Squad_Age            -0.01


In [47]:
t21.round(4).to_csv(PROCESSED / "model_task21_goal_diff.csv", index=False)
t22.round(4).to_csv(PROCESSED / "model_task22_goals_scored.csv", index=False)
print(f"Saved model_task21_goal_diff.csv    -> {t21.shape[0]} rows, {t21.shape[1] - 4} explanatory variables + target")
print(f"Saved model_task22_goals_scored.csv -> {t22.shape[0]} rows, {t22.shape[1] - 4} explanatory variables + target")

Saved model_task21_goal_diff.csv    -> 104 rows, 8 explanatory variables + target
Saved model_task22_goals_scored.csv -> 208 rows, 8 explanatory variables + target
